# GridLeak AI
## 02 — Ön işleme ve özellik üretimi

Bu defter abone bölmesini korur, takvimi düzenler, 2016 başlangıç denemesi için yedi özellik üretir ve eğitim/test çıktılarını kaydeder.
Model eğitimi, hiperparametre seçimi, SMOTE ve özellik öneminin değerlendirilmesi 03–05 defterlerinde yapılacaktır.
Doldurma/ölçekleme burada yalnız tanımlanır; her çapraz doğrulama eğitim katında modelle birlikte öğrenilmelidir.
2016 ve %80 dönem doluluğu başlangıç seçimleridir, en iyi oldukları henüz gösterilmemiştir.


### 1. Veriyi okuma ve kimlik/hedef ayrımı

In [1]:
from pathlib import Path
import pandas as pd

# Proje ve veri dosyasının konumu.
# Notebook hangi bilgisayarda açılırsa açılsın proje kökünü bulur.
proje_klasoru = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "data" / "raw" / "data.csv").is_file()),
    None
)
if proje_klasoru is None:
    raise FileNotFoundError("Notebook'u GridLeakAI/ai/notebooks içinden açın.")
veri_yolu = proje_klasoru / "data" / "raw" / "data.csv"

# Orijinal veriyi bu defterde yeniden okuyoruz.
df = pd.read_csv(veri_yolu, dtype={"CONS_NO": str})

# Abone kimliklerini, hedef etiketleri ve tüketimleri ayırıyoruz.
abone_kimlikleri = df["CONS_NO"].copy()
y = df["FLAG"].copy()
X = df.drop(columns=["CONS_NO", "FLAG"]).copy()

print("Abone sayısı:", len(df))
print("Tüketim sütunu sayısı:", X.shape[1])
print("Hedef etiket dağılımı:")
print(y.value_counts().sort_index())

from IPython.display import display
assert df["CONS_NO"].notna().all() and df["CONS_NO"].is_unique
assert y.notna().all() and set(y.unique()) == {0, 1}, "Hedef etiketleri kontrol edin."

Abone sayısı: 42372
Tüketim sütunu sayısı: 1034
Hedef etiket dağılımı:
FLAG
0    38757
1     3615
Name: count, dtype: int64


### 2. Abone bazında %80 eğitim / %20 test

In [2]:
from sklearn.model_selection import train_test_split

# Abonelerin satır indekslerini eğitim ve test olarak ayırıyoruz.
egitim_indeksleri, test_indeksleri = train_test_split(
    df.index,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Eğitim grubunun tüketimleri ve doğru cevapları.
X_train = X.loc[egitim_indeksleri].copy()
y_train = y.loc[egitim_indeksleri].copy()

# Test grubunun tüketimleri ve değerlendirmede kullanılacak cevapları.
X_test = X.loc[test_indeksleri].copy()
y_test = y.loc[test_indeksleri].copy()

# Abone kimliklerini takip için ayrı tutuyoruz.
egitim_aboneleri = abone_kimlikleri.loc[egitim_indeksleri].copy()
test_aboneleri = abone_kimlikleri.loc[test_indeksleri].copy()

print("Toplam abone:", len(df))
print("Eğitim abonesi:", len(X_train))
print("Test abonesi:", len(X_test))

# Aynı abone iki grupta birden bulunmamalı.
ortak_aboneler = set(egitim_aboneleri) & set(test_aboneleri)
print("İki grupta ortak abone sayısı:", len(ortak_aboneler))

# Sınıfların iki gruba nasıl dağıldığını gösteriyoruz.
dagilim = pd.DataFrame({
    "Eğitim abone sayısı": y_train.value_counts(),
    "Test abone sayısı": y_test.value_counts(),
    "Eğitim oranı (%)": y_train.value_counts(normalize=True) * 100,
    "Test oranı (%)": y_test.value_counts(normalize=True) * 100
}).sort_index()

display(dagilim.round(2))
assert not ortak_aboneler, "Eğitim ve test aboneleri kesişiyor."

Toplam abone: 42372
Eğitim abonesi: 33897
Test abonesi: 8475
İki grupta ortak abone sayısı: 0


,Eğitim abone sayısı,Test abone sayısı,Eğitim oranı (%),Test oranı (%)
FLAG,,,,
0,31005,7752,91.47,91.47
1,2892,723,8.53,8.53


### 3. Bölme listesini saklama

In [3]:
# Bölme listesinin kaydedileceği klasör.
splits_klasoru = proje_klasoru / "data" / "splits"
splits_klasoru.mkdir(parents=True, exist_ok=True)

# Eğitim abonelerini etiketliyoruz.
egitim_listesi = pd.DataFrame({
    "CONS_NO": egitim_aboneleri.to_numpy(),
    "grup": "egitim"
})

# Test abonelerini etiketliyoruz.
test_listesi = pd.DataFrame({
    "CONS_NO": test_aboneleri.to_numpy(),
    "grup": "test"
})

# İki listeyi tek tabloda birleştiriyoruz.
bolme_listesi = pd.concat(
    [egitim_listesi, test_listesi],
    ignore_index=True
)

# Her abonenin listede yalnızca bir kez bulunduğunu kontrol ediyoruz.
assert bolme_listesi["CONS_NO"].is_unique
assert len(bolme_listesi) == len(df)

# Mevcut farklı bir bölme listesinin üzerine yazmayı önlüyoruz.
kayit_yolu = splits_klasoru / "abone_bolunmesi.csv"

if kayit_yolu.exists():
    mevcut_liste = pd.read_csv(
        kayit_yolu,
        dtype={"CONS_NO": str}
    )

    pd.testing.assert_frame_equal(
        mevcut_liste.sort_values("CONS_NO").reset_index(drop=True),
        bolme_listesi.sort_values("CONS_NO").reset_index(drop=True)
    )

    print("Aynı bölme listesi zaten kayıtlı; değiştirilmedi.")
else:
    bolme_listesi.to_csv(
        kayit_yolu,
        index=False,
        encoding="utf-8-sig"
    )
    print("Bölme listesi kaydedildi.")

print("Dosyanın konumu:", kayit_yolu)
print(bolme_listesi["grup"].value_counts())

Aynı bölme listesi zaten kayıtlı; değiştirilmedi.
Dosyanın konumu: C:\Users\kadir\Desktop\GridLeakAI\data\splits\abone_bolunmesi.csv
grup
egitim    33897
test       8475
Name: count, dtype: int64


### 4. Takvimi sıralama ve eksik günü ekleme

In [4]:
# Eğitim tablosunun ayrı bir kopyasında çalışıyoruz.
X_train_takvim = X_train.copy()

# Metin biçimindeki sütun adlarını tarihe dönüştürüyoruz.
X_train_takvim.columns = pd.to_datetime(
    X_train_takvim.columns,
    format="%Y/%m/%d"
)

# Sütunları takvim sırasına koyuyoruz.
X_train_takvim = X_train_takvim.sort_index(axis=1)

# İlk ve son tarih arasındaki eksiksiz takvimi oluşturuyoruz.
tam_takvim = pd.date_range(
    start=X_train_takvim.columns.min(),
    end=X_train_takvim.columns.max(),
    freq="D"
)

# Eklenecek tarihleri belirliyoruz.
eklenen_tarihler = tam_takvim.difference(
    X_train_takvim.columns
)

# Eksik tarih sütunlarını NaN, yani bilinmeyen değerlerle ekliyoruz.
X_train_takvim = X_train_takvim.reindex(columns=tam_takvim)

print("Eğitim abonesi:", len(X_train_takvim))
print("Önceki tarih sütunu sayısı:", X_train.shape[1])
print("Yeni tarih sütunu sayısı:", X_train_takvim.shape[1])
print(
    "Eklenen tarihler:",
    eklenen_tarihler.strftime("%Y-%m-%d").tolist()
)

Eğitim abonesi: 33897
Önceki tarih sütunu sayısı: 1034
Yeni tarih sütunu sayısı: 1035
Eklenen tarihler: ['2016-09-18']


### 5. 2016 başlangıç dönemi ve doluluk

In [5]:
# İlk model denemesi için 2016 dönemini ayrı tutuyoruz.
X_train_2016 = X_train_takvim.loc[
    :, "2016-01-01":"2016-10-31"
].copy()

# Her abonenin kaç günlük gerçek kaydı bulunduğunu sayıyoruz.
egitim_dolu_gun = X_train_2016.count(axis=1)

# Takvimdeki günlerin yüzde kaçında kayıt bulunduğunu hesaplıyoruz.
egitim_doluluk = (
    egitim_dolu_gun / X_train_2016.shape[1] * 100
)

print("Eğitim abonesi:", len(X_train_2016))
print("Takvim günü sayısı:", X_train_2016.shape[1])
print(
    "Ortalama kayıt doluluğu (%):",
    round(egitim_doluluk.mean(), 2)
)
print(
    "Bu dönemde hiç kaydı olmayan eğitim abonesi:",
    (egitim_dolu_gun == 0).sum()
)

Eğitim abonesi: 33897
Takvim günü sayısı: 305
Ortalama kayıt doluluğu (%): 98.05
Bu dönemde hiç kaydı olmayan eğitim abonesi: 39


### 6. İlk beş özellik

In [6]:
# Hiç kaydı olmayan abonelerin paydasını boş bırakıyoruz.
# Böylece sıfıra bölme yapmıyoruz.
gecerli_gun_sayisi = egitim_dolu_gun.where(
    egitim_dolu_gun > 0
)

# Her abone için temel tüketim özelliklerini oluşturuyoruz.
ozellikler_train = pd.DataFrame(
    index=X_train_2016.index
)

ozellikler_train["ortalama_tuketim"] = (
    X_train_2016.mean(axis=1)
)

ozellikler_train["ortanca_tuketim"] = (
    X_train_2016.median(axis=1)
)

ozellikler_train["tuketim_standart_sapma"] = (
    X_train_2016.std(axis=1)
)

ozellikler_train["en_yuksek_tuketim"] = (
    X_train_2016.max(axis=1)
)

ozellikler_train["sifir_gun_orani"] = (
    X_train_2016.eq(0).sum(axis=1)
    / gecerli_gun_sayisi
)

# Veri kalitesini tüketim özelliklerinden ayrı tutuyoruz.
veri_kalitesi_train = pd.DataFrame({
    "dolu_gun_sayisi": egitim_dolu_gun,
    "doluluk_yuzdesi": egitim_doluluk
})

print("Özellik tablosu — abone, özellik:", ozellikler_train.shape)

print("\nİlk 5 abonenin özellikleri:")
display(ozellikler_train.head().round(3))

print("\nHer özellikte hesaplanamayan değer sayısı:")
print(ozellikler_train.isna().sum())

Özellik tablosu — abone, özellik: (33897, 5)

İlk 5 abonenin özellikleri:

Her özellikte hesaplanamayan değer sayısı:
ortalama_tuketim          39
ortanca_tuketim           39
tuketim_standart_sapma    39
en_yuksek_tuketim         39
sifir_gun_orani           39
dtype: int64


,ortalama_tuketim,ortanca_tuketim,tuketim_standart_sapma,en_yuksek_tuketim,sifir_gun_orani
34508,2.998,2.260,2.058,10.79,0.000
35750,9.315,8.635,4.703,36.32,0.000
28171,0.000,0.000,0.000,0.00,1.000
19428,9.782,9.115,2.477,23.03,0.000
15070,3.603,0.700,6.525,31.80,0.036


### 7. Dönemsel değişim özelliği

In [7]:
# Karşılaştıracağımız iki dönemi seçiyoruz.
ilk_donem = X_train_2016.loc[:, "2016-01-01":"2016-03-31"]
son_donem = X_train_2016.loc[:, "2016-08-01":"2016-10-31"]

# Her dönemde kayıtların ne kadarının dolu olduğunu hesaplıyoruz.
ilk_doluluk = ilk_donem.notna().mean(axis=1)
son_doluluk = son_donem.notna().mean(axis=1)

# İlk deneme kuralı:
# İki dönemde de en az %80 doluluk varsa değişimi hesaplayacağız.
yeterli_kayit = (ilk_doluluk >= 0.80) & (son_doluluk >= 0.80)

# Boş günleri sıfır saymadan dönem ortalamalarını hesaplıyoruz.
ilk_ortalama = ilk_donem.mean(axis=1)
son_ortalama = son_donem.mean(axis=1)

# İlk ortalama sıfırsa yüzde değişim tanımsızdır.
# Bu abonelerde paydayı boş bırakıyoruz.
gecerli_baslangic = ilk_ortalama.where(ilk_ortalama > 0)

degisim_yuzdesi = (
    (son_ortalama - ilk_ortalama)
    / gecerli_baslangic
    * 100
)

# Yeni özelliği tabloya ekliyoruz.
# Yeterli kaydı olmayanlarda değer boş kalır.
ozellikler_train["donemsel_degisim_yuzdesi"] = (
    degisim_yuzdesi.where(yeterli_kayit)
)

print("Özellik tablosunun boyutu:", ozellikler_train.shape)

print(
    "Dönemsel değişimi hesaplanan abone:",
    ozellikler_train["donemsel_degisim_yuzdesi"].notna().sum()
)

print(
    "Dönemsel değişimi hesaplanamayan abone:",
    ozellikler_train["donemsel_degisim_yuzdesi"].isna().sum()
)

display(
    ozellikler_train[
        ["ortalama_tuketim", "donemsel_degisim_yuzdesi"]
    ].head().round(2)
)

Özellik tablosunun boyutu: (33897, 6)
Dönemsel değişimi hesaplanan abone: 29808
Dönemsel değişimi hesaplanamayan abone: 4089


,ortalama_tuketim,donemsel_degisim_yuzdesi
34508,3.00,98.27
35750,9.31,-36.52
28171,0.00,NaN
19428,9.78,-7.44
15070,3.60,512.60


### 8. Kesintisiz sıfır gün dizisi

In [8]:
# Bir abonenin en uzun kesintisiz sıfır tüketim süresini hesaplar.
def en_uzun_sifir_serisi(satir):
    # Hiç kayıt yoksa sıfır gün süresini bilmiyoruz.
    if satir.notna().sum() == 0:
        return float("nan")

    mevcut_seri = 0
    en_uzun_seri = 0

    # Takvim sırasındaki günlük değerleri tek tek inceliyoruz.
    for deger in satir:
        if pd.notna(deger) and deger == 0:
            mevcut_seri += 1
            en_uzun_seri = max(en_uzun_seri, mevcut_seri)
        else:
            # Pozitif tüketim veya eksik kayıt diziyi keser.
            mevcut_seri = 0

    return en_uzun_seri


# Fonksiyonu her eğitim abonesine uygulayıp yeni sütunu ekliyoruz.
ozellikler_train["en_uzun_sifir_serisi"] = (
    X_train_2016.apply(en_uzun_sifir_serisi, axis=1)
)

print("Özellik tablosunun boyutu:", ozellikler_train.shape)

display(
    ozellikler_train[
        ["sifir_gun_orani", "en_uzun_sifir_serisi"]
    ].head(10).round(3)
)

print(
    "Sıfır serisi hesaplanamayan abone:",
    ozellikler_train["en_uzun_sifir_serisi"].isna().sum()
)

Özellik tablosunun boyutu: (33897, 7)
Sıfır serisi hesaplanamayan abone: 39


,sifir_gun_orani,en_uzun_sifir_serisi
34508,0.000,0.0
35750,0.000,0.0
28171,1.000,188.0
19428,0.000,0.0
15070,0.036,1.0
16180,0.003,1.0
719,0.000,0.0
38353,0.000,0.0
18219,0.000,0.0
15519,0.678,46.0


### 9. Hiç kaydı olmayanları ayrı tutma

In [9]:
# En az bir dolu tüketim kaydı bulunan aboneleri belirliyoruz.
kaydi_var = egitim_dolu_gun > 0

# Hiç kaydı olmayan aboneleri ayrı bir listede tutuyoruz.
veri_yetersiz_train = pd.DataFrame({
    "CONS_NO": egitim_aboneleri.loc[~kaydi_var],
    "neden": "2016 döneminde tüketim kaydı bulunmuyor"
})

# İlk model denemesinde kullanılacak özellikler ve hedefler.
X_train_model = ozellikler_train.loc[kaydi_var].copy()
y_train_model = y_train.loc[X_train_model.index].copy()

# Özellik satırları ile hedeflerin aynı sırada olduğunu kontrol ediyoruz.
assert X_train_model.index.equals(y_train_model.index)

print("Başlangıçtaki eğitim abonesi:", len(ozellikler_train))
print("Veri yetersiz grubundaki abone:", len(veri_yetersiz_train))
print("Model eğitimine alınacak abone:", len(X_train_model))
print("Özellik sayısı:", X_train_model.shape[1])

print("\nKalan özelliklerdeki eksik değer sayıları:")
print(X_train_model.isna().sum())

Başlangıçtaki eğitim abonesi: 33897
Veri yetersiz grubundaki abone: 39
Model eğitimine alınacak abone: 33858
Özellik sayısı: 7

Kalan özelliklerdeki eksik değer sayıları:
ortalama_tuketim               0
ortanca_tuketim                0
tuketim_standart_sapma         0
en_yuksek_tuketim              0
sifir_gun_orani                0
donemsel_degisim_yuzdesi    4050
en_uzun_sifir_serisi           0
dtype: int64


### 10. Özellik kalite kontrolü

In [10]:
import numpy as np

# Her özellikteki eksik ve sonsuz değerleri ayrı ayrı sayıyoruz.
ozellik_kontrol = pd.DataFrame({
    "Eksik değer sayısı": X_train_model.isna().sum(),
    "Eksik oranı (%)": X_train_model.isna().mean() * 100,
    "Sonsuz değer sayısı": np.isinf(X_train_model).sum(),
    "Farklı değer sayısı": X_train_model.nunique()
})

display(ozellik_kontrol.round(2))

,Eksik değer sayısı,Eksik oranı (%),Sonsuz değer sayısı,Farklı değer sayısı
ortalama_tuketim,0,0.00,0,31008
ortanca_tuketim,0,0.00,0,6417
tuketim_standart_sapma,0,0.00,0,31415
en_yuksek_tuketim,0,0.00,0,7612
sifir_gun_orani,0,0.00,0,2553
donemsel_degisim_yuzdesi,4050,11.96,0,28948
en_uzun_sifir_serisi,0,0.00,0,261


### 11. Dönemsel değişimin eksiklik nedenleri

In [11]:
# Önce iki dönemde yeterli kayıt bulunmayanları ayırıyoruz.
kayit_yetersiz = ~yeterli_kayit

# Yeterli kayıt bulunduğu hâlde ilk dönem ortalaması sıfır olanlar.
baslangic_sifir = yeterli_kayit & ilk_ortalama.eq(0)

print(
    "Dönemlerde yeterli kayıt olmadığı için hesaplanamayan:",
    kayit_yetersiz.loc[X_train_model.index].sum()
)

print(
    "Yeterli kayıt var ama ilk dönem ortalaması sıfır:",
    baslangic_sifir.loc[X_train_model.index].sum()
)

print(
    "Toplam eksik dönemsel değişim:",
    X_train_model["donemsel_degisim_yuzdesi"].isna().sum()
)

Dönemlerde yeterli kayıt olmadığı için hesaplanamayan: 1124
Yeterli kayıt var ama ilk dönem ortalaması sıfır: 2926
Toplam eksik dönemsel değişim: 4050


### 12. Doldurma ve ölçekleme planı

In [12]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Logistic Regression ve SVM için hazırlık zinciri.
on_isleme_olcekli = Pipeline([
    (
        "eksik_doldurma",
        SimpleImputer(
            strategy="median",
            add_indicator=True
        )
    ),
    (
        "olcekleme",
        StandardScaler()
    )
])

# Random Forest ve XGBoost için ölçeklemesiz hazırlık zinciri.
on_isleme_agac = Pipeline([
    (
        "eksik_doldurma",
        SimpleImputer(
            strategy="median",
            add_indicator=True
        )
    )
])

print("Ölçeklemeli ve ölçeklemesiz işlem zincirleri tanımlandı.")
print("Henüz doldurma veya ölçekleme uygulanmadı.")

Ölçeklemeli ve ölçeklemesiz işlem zincirleri tanımlandı.
Henüz doldurma veya ölçekleme uygulanmadı.


**Henüz fit uygulanmadı.** Eksiklik göstergesi önem puanı değildir. Ölçekleme LR/SVM içindir; ağaçlarda gerekli değildir.
Özellik seçimi burada yapılmadı: yedi aday özellik korunuyor. Katkıları ve SHAP açıklamaları eğitim/inceleme aşamalarında değerlendirilecek.
Sıfır kaydı olan aboneler “veri yetersiz” grubunda kalır; düşük ama sıfır olmayan doluluk tek başına güvenilirlik garantisi değildir.

### 13. Eğitim ve test için tek özellik fonksiyonu

Aşağıdaki fonksiyon yukarıda öğrendiğimiz yedi hesabı bir araya getirir. Yeni özellik eklemez.
Önce eğitim sonuçlarıyla aynı olduğunu kontrol ederiz; sonra testte aynı kuralları uygularız.
Test etiketleri özellik hesabına girmez, test başarısı burada ölçülmez.

In [13]:
def ozellikleri_hesapla(takvimli_tuketim):
    donem = takvimli_tuketim.loc[:, "2016-01-01":"2016-10-31"]
    dolu = donem.count(axis=1)
    payda = dolu.where(dolu > 0)
    sonuc = pd.DataFrame(index=donem.index)
    sonuc["ortalama_tuketim"] = donem.mean(axis=1)
    sonuc["ortanca_tuketim"] = donem.median(axis=1)
    sonuc["tuketim_standart_sapma"] = donem.std(axis=1)
    sonuc["en_yuksek_tuketim"] = donem.max(axis=1)
    sonuc["sifir_gun_orani"] = donem.eq(0).sum(axis=1) / payda
    ilk = donem.loc[:, "2016-01-01":"2016-03-31"]
    son = donem.loc[:, "2016-08-01":"2016-10-31"]
    yeterli = (ilk.notna().mean(axis=1) >= .80) & (son.notna().mean(axis=1) >= .80)
    ilk_ort = ilk.mean(axis=1)
    degisim = (son.mean(axis=1) - ilk_ort) / ilk_ort.where(ilk_ort > 0) * 100
    sonuc["donemsel_degisim_yuzdesi"] = degisim.where(yeterli)
    sonuc["en_uzun_sifir_serisi"] = donem.apply(en_uzun_sifir_serisi, axis=1)
    kalite = pd.DataFrame({"dolu_gun_sayisi": dolu, "doluluk_yuzdesi": dolu / donem.shape[1] * 100})
    return sonuc, kalite

kontrol_ozellik, kontrol_kalite = ozellikleri_hesapla(X_train_takvim)
pd.testing.assert_frame_equal(kontrol_ozellik, ozellikler_train)
pd.testing.assert_frame_equal(kontrol_kalite, veri_kalitesi_train)
assert not np.isinf(kontrol_ozellik.to_numpy()).any()
print("Ortak fonksiyon, adım adım hesaplanan yedi eğitim özelliğiyle aynı sonucu verdi.")

Ortak fonksiyon, adım adım hesaplanan yedi eğitim özelliğiyle aynı sonucu verdi.


In [14]:
# Test tüketimlerine eğitimdeki takvim ve aynı sabit özellik kuralları uygulanır.
X_test_takvim = X_test.copy()
X_test_takvim.columns = pd.to_datetime(X_test_takvim.columns, format="%Y/%m/%d")
X_test_takvim = X_test_takvim.sort_index(axis=1).reindex(columns=tam_takvim)
ozellikler_test, veri_kalitesi_test = ozellikleri_hesapla(X_test_takvim)
test_kaydi_var = veri_kalitesi_test["dolu_gun_sayisi"] > 0
X_test_model = ozellikler_test.loc[test_kaydi_var].copy()
y_test_model = y_test.loc[X_test_model.index].copy()
veri_yetersiz_test = pd.DataFrame({
    "CONS_NO": test_aboneleri.loc[~test_kaydi_var],
    "neden": "2016 döneminde tüketim kaydı bulunmuyor"
})
assert X_test_model.columns.equals(X_train_model.columns)
assert X_test_model.index.equals(y_test_model.index)
assert not np.isinf(X_test_model.to_numpy()).any()
assert len(X_test_model) + len(veri_yetersiz_test) == len(X_test)
print("Test hazırlığı tamamlandı. Etiket dağılımı veya model başarısı incelenmedi.")
print("Test toplamı:", len(X_test), "Skorlanabilir:", len(X_test_model), "Veri yetersiz:", len(veri_yetersiz_test))

Test hazırlığı tamamlandı. Etiket dağılımı veya model başarısı incelenmedi.
Test toplamı: 8475 Skorlanabilir: 8466 Veri yetersiz: 9


### 14. Çıktıların kaydedilmesi
Bu hücreler çalıştırıldığında aynı adlı üretilmiş dosyalar güncellenir. Ham veri ve abone bölmesi korunur. Eksik özellikler boş kalır. Kimlik ve FLAG, model girdilerinden ayrılmalıdır.

In [15]:
# Hazırlanan verilerin kaydedileceği klasör.
processed_klasoru = proje_klasoru / "data" / "processed"
processed_klasoru.mkdir(parents=True, exist_ok=True)

# Özellikler ile hedeflerin aynı abonelere karşılık geldiğini kontrol ediyoruz.
assert X_train_model.index.equals(y_train_model.index)

# Kaydetmek için ayrı bir tablo oluşturuyoruz.
egitim_kayit = X_train_model.copy()

# Abone kimliğini ve hedef etiketini ekliyoruz.
egitim_kayit.insert(
    0,
    "CONS_NO",
    abone_kimlikleri.loc[X_train_model.index]
)
egitim_kayit["FLAG"] = y_train_model

# Özellik tablosunu kaydediyoruz.
egitim_yolu = processed_klasoru / "egitim_ozellikleri_2016_v1.csv"

egitim_kayit.to_csv(
    egitim_yolu,
    index=False,
    encoding="utf-8-sig"
)

# Hiç kaydı olmadığı için eğitime alınmayan aboneleri de saklıyoruz.
yetersiz_yolu = processed_klasoru / "egitim_veri_yetersiz_2016.csv"

veri_yetersiz_train.to_csv(
    yetersiz_yolu,
    index=False,
    encoding="utf-8-sig"
)

print("Eğitim dosyası:", egitim_yolu)
print("Kaydedilen eğitim abonesi:", len(egitim_kayit))
print("Model özelliği sayısı:", X_train_model.shape[1])
print("Veri yetersiz abone sayısı:", len(veri_yetersiz_train))

Eğitim dosyası: C:\Users\kadir\Desktop\GridLeakAI\data\processed\egitim_ozellikleri_2016_v1.csv
Kaydedilen eğitim abonesi: 33858
Model özelliği sayısı: 7
Veri yetersiz abone sayısı: 39


In [16]:
# Test özellikleri ve etiketleri ayrı dosyalarda saklanır.
# 03 defteri test dosyalarını yöntem seçimi için okumamalıdır.
test_kayit = X_test_model.copy()
test_kayit.insert(0, "CONS_NO", abone_kimlikleri.loc[X_test_model.index])
test_kayit.to_csv(processed_klasoru / "test_ozellikleri_2016_v1.csv", index=False, encoding="utf-8-sig")

# Bütün test abonelerinin etiketleri korunur: veri yetersizler başarı raporunda gizlenmemeli.
test_hedefler = pd.DataFrame({
    "CONS_NO": test_aboneleri,
    "FLAG": y_test,
    "skorlanabilir": test_kaydi_var
})
test_hedefler.to_csv(processed_klasoru / "test_hedefleri_2016_v1.csv", index=False, encoding="utf-8-sig")
veri_yetersiz_test.to_csv(processed_klasoru / "test_veri_yetersiz_2016.csv", index=False, encoding="utf-8-sig")

# Kayıt kalitesini model özelliklerinden ayrı saklıyoruz.
for grup_adi, kalite in [("egitim", veri_kalitesi_train), ("test", veri_kalitesi_test)]:
    kayit = kalite.copy()
    kayit.insert(0, "CONS_NO", abone_kimlikleri.loc[kalite.index])
    kayit.to_csv(processed_klasoru / f"{grup_adi}_veri_kalitesi_2016.csv", index=False, encoding="utf-8-sig")

# Özellik sırası ve deney kuralları sonraki defterlerde izlenebilir olsun.
import json
ayarlar = {
    "surum": "2016_v1", "donem_baslangic": "2016-01-01", "donem_bitis": "2016-10-31",
    "test_orani": 0.20, "random_state": 42, "donemsel_doluluk_esigi": 0.80,
    "ozellikler": X_train_model.columns.tolist(),
    "kimlik": "CONS_NO", "hedef": "FLAG", "ogrenilmis_donusum_uygulandi": False,
    "not": "2016 başlangıç denemesi; dönem karşılaştırması henüz yapılmadı. İlk EDA tüm veri üzerinde yapıldı."
}
(processed_klasoru / "ozellik_tanimi_2016_v1.json").write_text(
    json.dumps(ayarlar, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("Eğitim/test özellikleri, test hedefleri, kalite tabloları ve deney tanımı kaydedildi.")

Eğitim/test özellikleri, test hedefleri, kalite tabloları ve deney tanımı kaydedildi.


## 15. Bu defterin sonucu ve sonraki adım

- Abone bazında 33.897 eğitim / 8.475 test ayrımı korunmuştur; ortak abone yoktur.
- Eksik 18 Eylül 2016 günü NaN olarak takvime eklenmiştir. Günlük tüketim boşlukları sıfır yapılmamıştır.
- 2016 için yedi aday özellik hesaplanmıştır. Hiç kaydı olmayan 39 eğitim abonesi ayrı listede tutulur; 33.858 abone eğitim girdisidir.
- Dönemsel değişimin 4.050 eğitim değerinde eksiklik vardır: 1.124 kayıt yetersizliği, 2.926 sıfır başlangıç ortalaması.
- Testte aynı kurallar uygulanır. Veri yetersiz test aboneleri son raporda toplam kapsamdan çıkarılarak gizlenmemelidir; hem skorlanan başarı hem kapsam oranı raporlanmalıdır.
- Doldurma ve ölçekleme **öğrenilmedi**; 03'te modelle aynı Pipeline içinde her CV eğitim katında öğrenilecektir.
- Özellik seçimi, SMOTE, dört modelin eğitimi ve hiperparametre aramaları henüz yapılmadı. Bunlar sonraki defterlerin görevidir.
- İlk EDA tüm veri üzerinde yapıldı; bu sınırlılık korunmuştur. Bu aşamadan sonra test, yöntem veya eşik seçmek için kullanılmaz.

**Burada durun. Sonraki çalışma: 03_model_training_and_tuning.ipynb.**
